In [2]:
import pandas as pd
import numpy as np
import networkx as nx
from sklearn.preprocessing import MinMaxScaler, normalize
from sklearn.neighbors import NearestNeighbors
import random
from google.colab import drive

# Load data
drive.mount('/content/drive')
file_path = '/content/drive/My Drive/Music Info.csv'
df = pd.read_csv(file_path, encoding='utf-8', on_bad_lines='skip')

# Remove blank spaces from dataset column names and store the column names
df.columns = df.columns.str.strip()

id_col = 'track_id'
name_col = 'name'
artist_col = 'artist'

if 'genre' in df.columns:
    genre_col = 'genre'
else:
    genre_col = None

# Remove rows with missing track ID, song name, or artist, filters out duplicate track IDs, and reorder the remaining IDs
df = df.dropna(subset=[id_col, name_col, artist_col])
df = df.drop_duplicates(subset=[id_col])
df = df.reset_index(drop=True)

TOTAL_SONGS = len(df)

# Create list of audio features from the dataset
audio_features = [
    'danceability',
    'energy',
    'loudness',
    'speechiness',
    'acousticness',
    'instrumentalness',
    'liveness',
    'valence',
    'tempo',
    'mode'
]

# Store features that are actually found in the dataset
available_features = []
for feature in audio_features:
    if feature in df.columns:
        available_features.append(feature)

print(f"The features being used for this recommendation are: {available_features}")

# Fills in missing data points based on median values
feature_medians = df[available_features].median()
df[available_features] = df[available_features].fillna(feature_medians)

# Scale feature scores to ensure proper vectoe distance calculations
scaled_features = MinMaxScaler().fit_transform(df[available_features])

# Normalize vectors and establish KNN parameters
normalized_features = normalize(scaled_features, norm='l2')
K_NEIGHBORS = 5
total_neighbors_to_find = K_NEIGHBORS + 1

# Establish KNN model and fit it using our normalized features
nn_model = NearestNeighbors(n_neighbors=total_neighbors_to_find, metric='euclidean', algorithm='kd_tree')
nn_model.fit(normalized_features)

# Obtain the indices of the nearest songs and their associated Euclidean distances from the input song
distances, indices = nn_model.kneighbors(normalized_features)

# Initialize the empty graph network
G = nx.Graph()

# Iterate through each song index and add a node to the graph for that song and artist
for idx in range(TOTAL_SONGS):
    G.add_node(idx, title=f"{df.iloc[idx][name_col]} by {df.iloc[idx][artist_col]}")

# Iterate through all indices of songs, calculate cosine similarity score of input song and indexed song,
# and add score boosts of the indexed song matches the artist and/or genre of the input song.
for global_idx in range(TOTAL_SONGS):
    target_artist = df.iloc[global_idx][artist_col]
    target_genre = df.iloc[global_idx][genre_col] if genre_col else None

    for neighbor_rank in range(1, K_NEIGHBORS + 1):
        match_idx = indices[global_idx][neighbor_rank]
        dist = distances[global_idx][neighbor_rank]

        # Calculate cosine similarity score from the Euclidean distance
        score = 1.0 - ((dist ** 2) / 2.0)

        if target_artist == df.iloc[match_idx][artist_col]:
            score += 0.05

        if genre_col and target_genre == df.iloc[match_idx][genre_col]:
            score += 0.03

        final_weight = min(1.0, float(score))

        # Updates graph edges if they already exist and ensures a maximum possible score of 1.0
        if G.has_edge(global_idx, match_idx):
            existing_weight = G[global_idx][match_idx]['weight']
            G[global_idx][match_idx]['weight'] = max(existing_weight, final_weight)
        else:
            G.add_edge(global_idx, match_idx, weight=final_weight)

print(f"Network Map Stats: {G.number_of_nodes():,} nodes | {G.number_of_edges():,} paths.")

# Define the search engine for the input song or artist
# Note**: This search will choose the lowest-indexed song containing the input term
# Takes the index of the chosen song and provides it to our recommendation engine
def find_song_by_search(search_term, dataframe, graph):
    matches = dataframe[
        dataframe[name_col].str.contains(search_term, case=False, na=False) |
        dataframe[artist_col].str.contains(search_term, case=False, na=False)
    ]

    if matches.empty:
        print(f"'{search_term}' was not found in the database.")
        return

    first_match_idx = matches.index[0] # ** This is where the matching with the lowest-indexed result occurs
    get_graph_recommendations(song_index=first_match_idx, graph=graph)

# Define the recommendation engine
# Ensure the song exists, obtain the neighbors that are one node away from the input song,
# sorts them based on edge weight, and displays the top recommendations
def get_graph_recommendations(song_index, graph, num_recommendations=5):
    if song_index not in graph:
        print("Track node index value is not in the graph.")
        return

    neighbors = list(graph.neighbors(song_index))
    neighbors_sorted = sorted(neighbors, key=lambda n: graph[song_index][n]['weight'], reverse=True)

    print(f"\n[ Target Track ]: '{graph.nodes[song_index]['title']}'")

    for rank, neighbor in enumerate(neighbors_sorted[:num_recommendations], 1):
        score = graph[song_index][neighbor]['weight']
        print(f"Recommendation {rank}: {graph.nodes[neighbor]['title']} ({score:.2%} Match)")


# Obtain target songs through a random selection and a user input

# Random target song
print("\n--- Example for the random song selection ---")
random_idx = random.randint(0, TOTAL_SONGS - 1)
get_graph_recommendations(song_index=random_idx, graph=G)

# User input target song
print("\n--- Example for the user input song selection ---")
user_search = input("Enter a song, band, or artist: ").strip()

if user_search == "":
    print("You did not enter anything.")
else:
    find_song_by_search(search_term=user_search, dataframe=df, graph=G)

# Define the graph metrics evaluation function
# Outputs the total number of nodes and edges, average number of connections per song
# the graph density, the average edge weight, max and min degrees, and # of connected components
def print_graph_metrics(graph):
    num_nodes = graph.number_of_nodes()
    num_edges = graph.number_of_edges()

    degrees = [d for n, d in graph.degree()]
    avg_degree = sum(degrees) / num_nodes if num_nodes > 0 else 0
    max_degree = max(degrees)
    min_degree = min(degrees)
    density = nx.density(graph)

    weights = [data['weight'] for _, _, data in graph.edges(data=True)]
    avg_weight = sum(weights) / len(weights) if weights else 0

    # Obtains the number of self-contained "sub-networks" in the overall graph
    num_components = nx.number_connected_components(graph)

    print("=== GRAPH NETWORK METRICS ===")
    print(f"Total Nodes (Songs):    {num_nodes:,}")
    print(f"Total Edges (Paths):    {num_edges:,}")
    print(f"Average Connections:    {avg_degree:.2f} edges/song")
    print(f"Connected Sub-graphs:   {num_components:,}")
    print(f"Degree Range (Min/Max): {min_degree} to {max_degree} edges")
    print(f"Graph Density:          {density:.6f}")
    print(f"Average Similarity:     {avg_weight:.2%}")
    print("=============================\n")

print_graph_metrics(G)

# Evaluate the performance of our recommendation system
def evaluate_precision_at_k(graph, dataframe, k=5, sample_size=100):
    # Samples a random song from our graph
    sample_indices = random.sample(list(graph.nodes()), min(sample_size, len(graph)))

    precision_scores = []
    hits = 0

    for song_idx in sample_indices:
        target_artist = dataframe.iloc[song_idx][artist_col]
        target_genre = dataframe.iloc[song_idx][genre_col] if genre_col else None

        neighbors = list(graph.neighbors(song_idx))
        neighbors_sorted = sorted(neighbors, key=lambda n: graph[song_idx][n]['weight'], reverse=True)[:k]

        if not neighbors_sorted:
            continue

        # For each sampled song, determine how many times a recommendation is relevant (matches artist and/or genre)
        relevant_matches = 0
        for neighbor in neighbors_sorted:
            rec_artist = dataframe.iloc[neighbor][artist_col]
            rec_genre = dataframe.iloc[neighbor][genre_col] if genre_col else None

            if rec_artist == target_artist or (genre_col and rec_genre == target_genre):
                relevant_matches += 1

        # Determine precision of our model based on relevant recommendations
        precision = relevant_matches / k
        precision_scores.append(precision)

        if relevant_matches > 0:
            hits += 1
    # Outputs the average precision and the fraction of songs that had at a relevant recommendation
    mean_precision = sum(precision_scores) / len(precision_scores)
    hit_rate = hits / len(sample_indices)

    print(f"=== EVALUATION RESULTS (K={k}, Sample={len(sample_indices)}) ===")
    print(f"Mean Precision@: {mean_precision:.2%}")
    print(f"Hit Rate:        {hit_rate:.2%} (Percentage of songs with ≥1 valid match)")
    print("==================================================")

evaluate_precision_at_k(graph=G, dataframe=df, k=5, sample_size=200)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
The features being used for this recommendation are: ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'mode']
Network Map Stats: 50,683 nodes | 177,700 paths.

--- Example for the random song selection ---

[ Target Track ]: 'Antarctica by Immortal'
Recommendation 1: All That Remains by Bolt Thrower (99.93% Match)
Recommendation 2: Deme Quaden Thyrane by Marduk (99.90% Match)
Recommendation 3: Here Come The Rome Plows by Drive Like Jehu (99.87% Match)
Recommendation 4: The Shining by Anorexia Nervosa (99.86% Match)
Recommendation 5: The Final Massacre by Vader (99.85% Match)

--- Example for the user input song selection ---
Enter a song, band, or artist: Fleetwood Mac

[ Target Track ]: 'Go Your Own Way by Fleetwood Mac'
Recommendation 1: Pulling Mussels (From The Shell) by Squeeze (100